# Indian Sign Language: Mediapipe Holistic Landmarks Generation

In [1]:
!pip install -q --no-deps --upgrade mediapipe
!wget -q -O /tmp/holistic_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task
!cp /kaggle/input/datasets/swaptr/isl-video/label_map.json /kaggle/working/label_map.json

import glob, json, time
from pathlib import Path
from multiprocessing import Pool, cpu_count
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_PATH      = "/tmp/holistic_landmarker.task"
INPUT_GLOB      = "/kaggle/input/datasets/swaptr/isl-video/**/*"
OUTPUT_ROOT     = Path("/kaggle/working/keypoints")
SUPPORTED_EXT   = (".mov", ".mp4")
DOWNSCALE_WIDTH = 480
FRAME_STRIDE    = 1

FACE_COUNT, POSE_COUNT, HAND_COUNT = 478, 33, 21
PER_FRAME_TOTAL = FACE_COUNT + HAND_COUNT + POSE_COUNT + HAND_COUNT
TYPE_NAMES = np.array(["face", "left_hand", "pose", "right_hand"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 45.7 MB/s eta 0:00:00


2026-05-22 01:23:44.908247: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779413025.189970      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779413025.271514      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779413025.940458      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779413025.940501      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779413025.940504      16 computation_placer.cc:177] computation placer alr

In [2]:
_LANDMARKER = None

def _init_worker(model_path):
    global _LANDMARKER
    _LANDMARKER = mp_vision.HolisticLandmarker.create_from_options(
        mp_vision.HolisticLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=model_path),
            running_mode=mp_vision.RunningMode.VIDEO,
        )
    )

In [3]:
def _write_block(landmarks, type_id, count, base, buf, frame_idx, fy):
    sl = slice(base, base + count)
    if landmarks:
        buf['x'][sl] = np.fromiter((lm.x for lm in landmarks), dtype=np.float32, count=count)
        buf['y'][sl] = np.fromiter((lm.y for lm in landmarks), dtype=np.float32, count=count) * fy
        buf['z'][sl] = np.fromiter((lm.z for lm in landmarks), dtype=np.float32, count=count)
    else:
        buf['x'][sl] = buf['y'][sl] = buf['z'][sl] = np.nan
    buf['frame'][sl]          = frame_idx
    buf['type_id'][sl]        = type_id
    buf['landmark_index'][sl] = np.arange(count, dtype=np.int32)


def process_one_video(args):
    src_path, out_path = args
    cap = cv2.VideoCapture(str(src_path))
    n_hint = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    cap_rows = max(n_hint, 1) * PER_FRAME_TOTAL
    buf = {k: np.empty(cap_rows, dtype=t) for k, t in [
        ('frame', np.int32), ('type_id', np.int8), ('landmark_index', np.int32),
        ('x', np.float32), ('y', np.float32), ('z', np.float32)]}

    offset, frame_no, raw_idx = 0, 0, 0
    while True:
        ok, image = cap.read()
        if not ok:
            break
        if raw_idx % FRAME_STRIDE != 0:
            raw_idx += 1
            continue

        h, w, _ = image.shape
        if DOWNSCALE_WIDTH and w > DOWNSCALE_WIDTH:
            new_h = int(h * DOWNSCALE_WIDTH / w)
            image = cv2.resize(image, (DOWNSCALE_WIDTH, new_h), interpolation=cv2.INTER_AREA)
            h, w = new_h, DOWNSCALE_WIDTH
        fy = h / w

        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        result = _LANDMARKER.detect_for_video(mp_img, time.monotonic_ns() // 1_000_000)

        face = result.face_landmarks       or None
        lh   = result.left_hand_landmarks  or None
        pose = result.pose_landmarks       or None
        rh   = result.right_hand_landmarks or None

        _write_block(face, 0, FACE_COUNT, offset, buf, frame_no, fy); offset += FACE_COUNT
        _write_block(lh,   1, HAND_COUNT, offset, buf, frame_no, fy); offset += HAND_COUNT
        _write_block(pose, 2, POSE_COUNT, offset, buf, frame_no, fy); offset += POSE_COUNT
        _write_block(rh,   3, HAND_COUNT, offset, buf, frame_no, fy); offset += HAND_COUNT

        frame_no += 1
        raw_idx  += 1
    cap.release()

    for k in buf:
        buf[k] = buf[k][:offset]

    type_str = TYPE_NAMES[buf['type_id']]
    row_id = np.char.add(
        np.char.add(buf['frame'].astype('U'), '-'),
        np.char.add(type_str, np.char.add('-', buf['landmark_index'].astype('U'))),
    )

    out_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        'frame':          buf['frame'],
        'row_id':         row_id,
        'type':           type_str,
        'landmark_index': buf['landmark_index'],
        'x':              buf['x'],
        'y':              buf['y'],
        'z':              buf['z'],
    }).to_parquet(out_path, index=False)
    return str(src_path)

In [4]:
label_set = set()
jobs = []
for fp in glob.glob(INPUT_GLOB, recursive=True):
    if not fp.lower().endswith(SUPPORTED_EXT):
        continue
    p = Path(fp)
    label_set.add(p.parent.name)
    out = OUTPUT_ROOT / p.parent.parent.name / p.parent.name / f"{p.stem}.parquet"
    if out.exists():
        continue
    jobs.append((fp, out))

print(f"{len(jobs)} videos | {cpu_count()} workers")

t0 = time.time()
with Pool(cpu_count(), initializer=_init_worker, initargs=(MODEL_PATH,)) as pool:
    for i, _ in enumerate(pool.imap_unordered(process_one_video, jobs, chunksize=1), 1):
        if i % 25 == 0:
            print(f"[{i}/{len(jobs)}] {time.time()-t0:.1f}s")
print(f"done in {time.time()-t0:.1f}s")

4284 videos | 4 workers


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779413052.439860      74 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779413052.441232      81 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779413052.443729      66 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779413052.445306      85 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779413052.477074      74 inference_

[25/4284] 44.8s
[50/4284] 84.7s
[75/4284] 131.2s
[100/4284] 173.9s
[125/4284] 211.7s
[150/4284] 258.5s
[175/4284] 296.6s
[200/4284] 339.1s
[225/4284] 381.8s
[250/4284] 424.7s
[275/4284] 464.0s
[300/4284] 501.4s
[325/4284] 542.1s
[350/4284] 579.7s
[375/4284] 620.0s
[400/4284] 662.0s
[425/4284] 706.7s
[450/4284] 752.1s
[475/4284] 803.4s
[500/4284] 844.1s
[525/4284] 887.5s
[550/4284] 926.4s
[575/4284] 965.0s
[600/4284] 1004.3s
[625/4284] 1049.1s
[650/4284] 1087.0s
[675/4284] 1131.6s
[700/4284] 1174.9s
[725/4284] 1216.6s
[750/4284] 1255.3s
[775/4284] 1296.8s
[800/4284] 1340.9s
[825/4284] 1388.4s
[850/4284] 1430.8s
[875/4284] 1474.4s
[900/4284] 1522.2s
[925/4284] 1564.8s
[950/4284] 1614.9s
[975/4284] 1671.6s
[1000/4284] 1732.0s
[1025/4284] 1796.0s
[1050/4284] 1853.4s
[1075/4284] 1910.0s
[1100/4284] 1962.5s
[1125/4284] 2024.4s
[1150/4284] 2089.9s
[1175/4284] 2159.7s
[1200/4284] 2227.3s
[1225/4284] 2280.6s
[1250/4284] 2332.1s
[1275/4284] 2388.2s
[1300/4284] 2437.7s
[1325/4284] 2487.3s
[1350/4